# 13 - Fairness-aware targeting

Notebook 11 built a targeting policy that ranks people by estimated benefit,
showed it captures most of the achievable gain, and then closed with a
limitation: such a policy "can amplify bias if the model inherits historical
inequities". That sentence was true and it was also unexamined, which is a
strange thing for a repository about not overclaiming.

This notebook examines it. A rule that uses no protected attribute — only
predicted effect — is often described as neutral. That describes its *inputs*.
The allocation it produces is a separate question, and the two scenarios below
show it can be uneven for two quite different reasons, with quite different
implications for what to do about it.

## Causal question

Under a fixed budget, who does a benefit-ranked targeting policy actually
treat — and if the allocation across groups is uneven, what would it cost to
even it out?

## Data and design

- **Unit of analysis:** one person eligible for a service.
- **Treatment:** `treatment`, scarce; the budget binds.
- **Outcome:** `outcome`.
- **Observable covariates:** `age`, `screening_score`, `prior_visits`. These are
  what the targeting model sees.
- **Group:** `group`, with group B (labelled 1) about 30% of the population.
  **The model never sees it.**
- **Ground truth:** `true_ite`, used only to evaluate.

The two groups are constructed to have the same distribution of underlying
need. What differs is how well that need is *measured*: group B records fewer
visits at the same severity, and its screening score is substantially noisier.

We run two scenarios. In the first, true benefit is identical across groups. In
the second, group A genuinely benefits more.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd

from causal_inference_lab import (
    TMetaLearner,
    allocation_disparity,
    allocation_table,
    make_service_allocation_population,
    parity_constrained_selection,
    top_k_selection,
)

COVARIATES = ["age", "screening_score", "prior_visits"]
BUDGET_SHARE = 0.2

equal_benefit = make_service_allocation_population(effect_gap=0.0, seed=31).data
data = equal_benefit

print(f"population: {len(data):,}   group B share: {data['group'].mean():.1%}")
print()
print("by group — need is the same, its measurement is not:")
print(
    data.groupby("group")
    .agg(
        true_severity=("true_severity", "mean"),
        true_effect=("true_ite", "mean"),
        prior_visits=("prior_visits", "mean"),
        screening_sd=("screening_score", "std"),
    )
    .to_string(float_format=lambda v: f"{v:.3f}")
)

population: 6,000   group B share: 30.6%

by group — need is the same, its measurement is not:
       true_severity  true_effect  prior_visits  screening_sd
group                                                        
0             -0.019        1.494         1.732         1.038
1             -0.008        1.498         1.280         2.447


**Interpretation.** Underlying severity is the same in both groups
(−0.02 against −0.01) and so is true benefit (1.494 against 1.498). What differs
is the record: group B averages 1.3 prior visits against 1.7, and its screening
score has more than twice the spread (2.45 against 1.04), because it is measured
with far more noise.

Nothing here is a difference in need. It is a difference in how legible that
need is to a model.

## Estimand

Two, and only the first is statistical.

The **policy value** — total true benefit realised by whoever is treated — as in
notebook 11.

The **allocation** each group receives: the share of that group treated. This is
not an estimand in the causal sense at all. It is a description of what the
policy does, and it needs measuring precisely because no causal quantity implies
it.

## Identification assumptions

The targeting model inherits everything from notebook 03 — conditional
ignorability, overlap, correctly represented effect heterogeneity. Two more
matter here:

1. **The measured covariates capture need equally well across groups.** This is
   the assumption the scenario is built to violate, and it is rarely stated,
   let alone checked.
2. **Equal allocation is a meaningful target.** Demographic parity is one
   fairness criterion among several. It can conflict with others, and choosing
   it is a normative decision, not a statistical one. Nothing in this notebook
   establishes that parity is correct — only what it costs.

## Estimation

Fit the CATE model on observables, then compare two policies at the same budget:
rank everyone by estimated benefit, or apply the same ranking within each group
under a proportional quota.

In [2]:
def targeting_report(frame: pd.DataFrame, label: str) -> dict[str, object]:
    """Fit a CATE model, apply both policies, and report what each allocates."""
    model = TMetaLearner().fit(
        frame, covariates=COVARIATES, treatment_col="treatment", outcome_col="outcome"
    )
    scores = model.predict_cate(frame[COVARIATES])
    groups = frame["group"].to_numpy()
    truth = frame["true_ite"].to_numpy()
    budget = int(BUDGET_SHARE * len(frame))

    unconstrained = top_k_selection(scores, budget)
    parity = parity_constrained_selection(scores, groups, budget)

    print(f"=== {label} ===")
    print(f"budget: {budget:,} of {len(frame):,}\n")
    print("benefit-ranked policy:")
    print(
        allocation_table(groups, unconstrained, true_effects=truth).to_string(
            index=False, float_format=lambda v: f"{v:.3f}"
        )
    )
    return {
        "scores": scores,
        "groups": groups,
        "truth": truth,
        "unconstrained": unconstrained,
        "parity": parity,
    }


equal = targeting_report(equal_benefit, "Scenario 1: identical true benefit")

=== Scenario 1: identical true benefit ===
budget: 1,200 of 6,000

benefit-ranked policy:
 group  population  population_share  treated  allocation_rate  share_of_treated  mean_true_effect  realised_benefit
     0        4162             0.694      770            0.185             0.642             1.494          1424.415
     1        1838             0.306      430            0.234             0.358             1.498           739.982


**Interpretation.** The two groups benefit identically — mean true effect 1.494
and 1.498 — and the policy treats them at different rates: 18.5% of group A
against 23.4% of group B. Group B is 30.6% of the population and receives 35.8%
of the treatment.

The direction is worth dwelling on, because it is the opposite of what the
framing invites. The under-measured group is *over*-selected here, not
under-selected. Noisy estimates are dispersed estimates: group B's predicted
effects spread further in both directions, so more of them land in the top 20%.
Had the noise entered differently — through a proxy the model leaned on rather
than through dispersion — the disparity would run the other way.

That instability is the practical finding. The disparity is not a stable
property of "being under-measured"; it is a property of how the measurement
error interacts with the ranking. Which is precisely why allocation has to be
measured rather than reasoned about.

## Diagnostics

The disparity measure, and what parity costs. Since the groups benefit
identically here, evening out allocation should cost nothing — that is a
testable claim.

In [3]:
def compare_policies(report: dict[str, object]) -> pd.DataFrame:
    """Disparity and realised benefit under each policy."""
    groups, truth = report["groups"], report["truth"]
    rows = []
    for name, selected in (("benefit-ranked", report["unconstrained"]), ("parity", report["parity"])):
        rows.append(
            {
                "policy": name,
                "allocation disparity": allocation_disparity(groups, selected),
                "total true benefit": float(truth[selected].sum()),
            }
        )
    table = pd.DataFrame(rows)
    baseline = table.loc[table["policy"] == "benefit-ranked", "total true benefit"].iloc[0]
    table["benefit vs ranked"] = table["total true benefit"] / baseline - 1.0
    return table


print(compare_policies(equal).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

        policy  allocation disparity  total true benefit  benefit vs ranked
benefit-ranked                0.0489           2164.3973             0.0000
        parity                0.0003           2169.0336             0.0021


**Interpretation.** The benefit-ranked policy has an allocation disparity of
0.049 — a five percentage point gap in treatment rates between groups. Parity
removes it exactly, by construction.

And it costs nothing. Total realised benefit is *higher* under parity: 2,169
against 2,164, a gain of 0.2%. That is not a rounding artefact with a tidy
explanation to be waved at — it follows from the setup. When two groups benefit
equally, a ranking that separates them is tracking estimation noise rather than
benefit, so constraining it discards noise rather than signal.

The uncomfortable version of this result: the "efficient" policy was not more
efficient. It was differently allocated, for reasons that had nothing to do with
who benefits.

That is the easy case. Now the case where the groups genuinely differ, which is
where the trade-off becomes real.

In [4]:
unequal_benefit = make_service_allocation_population(effect_gap=0.6, seed=31).data
unequal = targeting_report(unequal_benefit, "Scenario 2: group A benefits more")

print()
print(compare_policies(unequal).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

=== Scenario 2: group A benefits more ===
budget: 1,200 of 6,000

benefit-ranked policy:
 group  population  population_share  treated  allocation_rate  share_of_treated  mean_true_effect  realised_benefit
     0        4162             0.694      959            0.230             0.799             2.094          2277.500
     1        1838             0.306      241            0.131             0.201             1.498           435.335

        policy  allocation disparity  total true benefit  benefit vs ranked
benefit-ranked                0.0993           2712.8341             0.0000
        parity                0.0003           2651.2925            -0.0227


**Interpretation.** Now group A's true effect is 2.094 against group B's 1.498,
and the policy responds: 23.0% of group A treated against 13.1% of group B, so
group B receives 20.1% of the treatment while being 30.6% of the population.
The disparity doubles to 0.099.

Parity still removes it, and now it costs 2.27% of total benefit. That is a real
trade-off, and it is not one this notebook can resolve. Whether forgoing 2.3% of
aggregate benefit to treat both groups at equal rates is right depends on why
group A benefits more, whether that reason is itself the product of prior
inequity, and who is entitled to the service — none of which is a statistical
question.

What the analysis contributes is the exchange rate. "Parity costs 2.3% here"
is a fact a decision-maker can weigh. "Targeting may amplify bias" is not.

## Uncertainty

Both scenarios use one draw. The disparity and the cost of parity are
statistics like any other, so they have sampling variability — and a disparity
indistinguishable from zero should not be acted on.

In [5]:
def bootstrap_policy_gap(frame: pd.DataFrame, draws: int = 60, seed: int = 5) -> pd.DataFrame:
    """Resample the population and recompute both statistics."""
    rng = np.random.default_rng(seed)
    disparities, costs = [], []

    for _ in range(draws):
        sample = frame.iloc[rng.integers(0, len(frame), len(frame))].reset_index(drop=True)
        model = TMetaLearner().fit(
            sample, covariates=COVARIATES, treatment_col="treatment", outcome_col="outcome"
        )
        scores = model.predict_cate(sample[COVARIATES])
        groups = sample["group"].to_numpy()
        truth = sample["true_ite"].to_numpy()
        budget = int(BUDGET_SHARE * len(sample))

        ranked = top_k_selection(scores, budget)
        parity = parity_constrained_selection(scores, groups, budget)
        disparities.append(allocation_disparity(groups, ranked))
        costs.append(1.0 - truth[parity].sum() / truth[ranked].sum())

    return pd.DataFrame({"disparity": disparities, "parity_cost": costs})


for label, frame in (("identical benefit", equal_benefit), ("group A benefits more", unequal_benefit)):
    draws = bootstrap_policy_gap(frame)
    low, high = np.percentile(draws["disparity"], [2.5, 97.5])
    cost_low, cost_high = np.percentile(draws["parity_cost"], [2.5, 97.5])
    print(f"{label}:")
    print(f"  disparity   {draws['disparity'].mean():.3f}  95% [{low:.3f}, {high:.3f}]")
    print(f"  parity cost {draws['parity_cost'].mean():+.4f}  95% [{cost_low:+.4f}, {cost_high:+.4f}]")

identical benefit:
  disparity   0.062  95% [0.002, 0.159]
  parity cost -0.0042  95% [-0.0157, +0.0017]


group A benefits more:
  disparity   0.067  95% [0.010, 0.109]
  parity cost +0.0189  95% [-0.0039, +0.0354]


**Interpretation.** The two statistics behave differently, and the difference
matters.

**The disparity is real.** Both intervals exclude zero — [0.002, 0.159] and
[0.010, 0.109]. The uneven allocation is not a fluke of one sample.

**The cost of parity is not established in either scenario.** Both intervals
straddle zero: [−0.016, +0.002] with identical benefits, and [−0.004, +0.035]
when group A genuinely benefits more. The point estimates go in the directions
the earlier tables suggested — slightly negative, then +1.9% — and the mass of
the second sits positive, but 60 resamples on 6,000 people cannot establish that
parity costs anything at conventional confidence.

That is worth stating plainly rather than glossing, because the single-draw
figure of 2.27% reads far more decisively than it deserves to. A decision
resting on that number needs more data behind it.

One technical caution about the disparity interval. Disparity is a maximum minus
a minimum of non-negative rates, so it cannot go below zero and resampling
pushes it upward: the bootstrap mean of 0.062 exceeds the point estimate of
0.049 in scenario 1 for that reason alone. Treat these intervals as indicating
whether a disparity exists, not as unbiased estimates of its size.

## Limitations

- **Demographic parity is one criterion, chosen here for concreteness.** Equal
  allocation rates, equal benefit received, and equal treatment of equally
  needy individuals are different targets that can conflict. Nothing here
  argues parity is the right one.
- **Two groups, one attribute.** Real allocation questions involve several
  attributes at once, where parity on each can be unachievable simultaneously.
- **The direction of the measurement-driven disparity is not robust.** Scenario
  1 over-selects the under-measured group; a different noise channel reverses
  it. Do not read "under-measured groups get less" as the general rule — read
  "benefit ranking is not neutral with respect to measurement quality".
- **`true_ite` is available here and never is in practice.** Every cost figure
  depends on knowing the truth. A real audit must estimate policy value from a
  holdout experiment, with wider uncertainty than shown.
- **The model never sees `group`.** That is realistic for a protected attribute
  and is also why the disparity is invisible without an explicit audit. Fitting
  on group would raise separate legal and ethical questions not addressed here.
- **Parity is imposed by quota, which is a blunt instrument.** It equalises
  rates but ignores within-group need distributions; a constrained optimisation
  over a fairness-penalised objective would do better and is not implemented.
- **No feedback loop is modelled.** The serious version of this problem is
  dynamic: today's allocation shapes tomorrow's records, which shape tomorrow's
  model. A single-period analysis cannot capture that, and it is the mechanism
  by which measurement inequity compounds.